In [3]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader\

login(dotenv_values()['HUGGINGFACE_API_KEY'])

In [2]:
csv_path   = '../data/SEP-28k-Extended_clips.csv'
clips_path = '../data/SEP-28k_CLIP'

In [3]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


CSV exists: True
Data Exists: True


In [4]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

Total clips in CSV:     28177
Clips found on disk:    28177
Clips NOT found:        0

Sample filepath:
../data/SEP-28k_CLIP\HeStutters\0\HeStutters_0_0.wav


In [5]:
stutter_cols = ['Prolongation','Block','SoundRep','WordRep','Interjection']

def create_label(row):
  if any(row[col] >= 2 for col in stutter_cols):
    return 1
  elif row['NoStutteredWords'] >=2:
    return 0
  else:
    return -1
df['label'] = df.apply(create_label, axis=1)
df = df[df['label'] != -1]

print(f"Fluent clips:  {len(df[df['label'] == 0])}")
print(f"Stutter clips: {len(df[df['label'] == 1])}")

Fluent clips:  12419
Stutter clips: 14022


In [6]:
train_df = df[df['SEP28k-T']== 'train'].reset_index(drop=True)
valid_df = df[df['SEP28k-T']== 'dev'].reset_index(drop=True)
test_df = df[df['SEP28k-T']== 'test'].reset_index(drop=True)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [8]:
# load voc2vec Encoder
print("\n Loading voc2vec encoder...")
MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")


 Loading voc2vec encoder...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 707.68it/s]


In [9]:
def convo_audio_data(filename):
  try:
    waveform,sample_rate=torchaudio.load(filename)
    #ensures both input and encoder are on the same device
    waveform = waveform.to(device)
    # Stereo to mono
    if waveform.shape[0] > 1:
      waveform = torch.mean(waveform,dim=0,keepdim=True)
    # Resample
    if sample_rate != 16000:
      waveform = torchaudio.functional.sample(
          waveform,
          sample_rate,
          16000,
      )
    # extract features
    emission = encoder(waveform.to(device)).last_hidden_state
    emission = emission.detach().cpu().numpy()
    return emission
  except Exception as e:
    print(f"Error processing {filename}: {e}")
    return None

In [10]:
# Generate embeddings
def generate_embeddings(dataframe,desc):
  x=[]
  y=[]
  discarded = 0
  for _, row in tqdm(dataframe.iterrows(), desc=desc, total=len(dataframe)):
        embedding = convo_audio_data(row['filepath'])

        if embedding is None:
            discarded += 1
        elif ((np.shape(embedding)[0] != 1) |
              (np.shape(embedding)[1] != 149) |
              (np.shape(embedding)[2] != 768)):
            discarded += 1
        else:
            x.append(embedding)
            y.append(row['label'])

  print(f"Discarded in {desc}: {discarded}")
  return x, y

print("\nGenerating embeddings...")
data_start_time = datetime.now()

x_train, y_train = generate_embeddings(train_df, "Train")
x_valid, y_valid = generate_embeddings(valid_df, "Dev")
x_test,  y_test  = generate_embeddings(test_df,  "Test")

print("Time taken: " + str(datetime.now() - data_start_time))


Generating embeddings...


Train: 100%|██████████| 6205/6205 [01:43<00:00, 60.07it/s]


Discarded in Train: 6


Dev: 100%|██████████| 6010/6010 [01:40<00:00, 59.81it/s]


Discarded in Dev: 8


Test: 100%|██████████| 14226/14226 [03:59<00:00, 59.48it/s]

Discarded in Test: 7
Time taken: 0:07:22.950621


In [11]:
# Hyper Parameters
batch_size    = 256
num_epochs    = 150
learning_rate = 0.0001

n_samples_train = len(x_train)
n_samples_valid = len(x_valid)
n_samples_test  = len(x_test)

print(f'\nTrain samples: {n_samples_train}')
print(f'Dev samples:   {n_samples_valid}')
print(f'Test samples:  {n_samples_test}')


Train samples: 6199
Dev samples:   6002
Test samples:  14219


In [12]:
# Dataset Class
class AudioDataset(Dataset):
    def __init__(self, x, y, n_samples):
        self.x         = x
        self.y         = y
        self.n_samples = n_samples

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.n_samples

# Create Datasets & DataLoaders
train_dataset = AudioDataset(x_train, y_train, n_samples_train)
valid_dataset = AudioDataset(x_valid, y_valid, n_samples_valid)
test_dataset  = AudioDataset(x_test,  y_test,  n_samples_test)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size,
                          shuffle=True,  num_workers=0)
valid_loader = DataLoader(dataset=valid_dataset, batch_size=batch_size,
                          shuffle=True,  num_workers=0)
test_loader  = DataLoader(dataset=test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=0)

print("\n All data loaders ready!")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches:  {len(test_loader)}")


 All data loaders ready!
Train batches: 25
Valid batches: 24
Test batches:  56


In [13]:
# Model definition
import torch.nn as nn

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super(AttentionPooling, self).__init__()
        # Attention layer
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)                    # → (batch, time_steps, 1)
        weights = torch.softmax(scores, dim=1)         # → (batch, time_steps, 1)
        pooled  = torch.sum(x * weights, dim=1)        # → (batch, hidden_size)
        return pooled


class StutterNet(nn.Module):
    def __init__(self):
        super(StutterNet, self).__init__()

        # Layer 1 - Linear Projection
        # Reduces voc2vec output from 768 → 384
        self.projection = nn.Linear(768, 384)

        # Layer 2 - 1D CNN
        # Looks for local patterns in the audio
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=384, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 3 - 2 layers of BiLSTM
        # Looks at audio forward and backwards
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Layer 4 - Attention Pooling
        # Focuses on most important parts of audio
        self.attention = AttentionPooling(hidden_size=512)

        # Layer 5 - Dense Layers
        self.dense = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 6 - 5 Output Heads
        # One head per stutter type!
        self.head_prolongation = nn.Linear(128, 2)
        self.head_block        = nn.Linear(128, 2)
        self.head_soundrep     = nn.Linear(128, 2)
        self.head_wordrep      = nn.Linear(128, 2)
        self.head_interjection = nn.Linear(128, 2)

    def forward(self, x):
        # x shape coming in → (batch, 1, 149, 768)
        # Remove extra dimension
        x = x.squeeze(1)

        # Linear Projection
        x = self.projection(x)
        x = torch.relu(x)

        # 1D CNN
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        # BiLSTM
        x, _ = self.bilstm(x)

        # Attention Pooling
        x = self.attention(x)

        # Dense Layers
        x = self.dense(x)

        # 5 Output Heads
        output_prolongation = self.head_prolongation(x)
        output_block        = self.head_block(x)
        output_soundrep     = self.head_soundrep(x)
        output_wordrep      = self.head_wordrep(x)
        output_interjection = self.head_interjection(x)

        return {
            'prolongation': output_prolongation,
            'block':        output_block,
            'soundrep':     output_soundrep,
            'wordrep':      output_wordrep,
            'interjection': output_interjection
        }


# Initialize Model
model  = StutterNet().to(device)
print(model)

StutterNet(
  (projection): Linear(in_features=768, out_features=384, bias=True)
  (cnn): Sequential(
    (0): Conv1d(384, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (bilstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): AttentionPooling(
    (attention): Linear(in_features=512, out_features=1, bias=True)
  )
  (dense): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
  )
  (head_prolongation): Linear(in_features=128, out_features=2, bias=True)
  (head_block): Linear(in_features=128, out_features=2, bias=True)
  (head_soundrep): Linear(in_features=128, out_features=2, bias=True)
  (head_wordrep): Linear(in_features=128, out_features=2, bias=True)
  (head_interjection): Linear(

In [17]:
from torch.utils.tensorboard import SummaryWriter
# early_stopper = EarlyStopping(patience=5, delta=0.01)
writer = SummaryWriter()

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_accu= []
train_losses = []
eval_accu=[]
eval_losses=[]

def train(epoch):
        print('\nEpoch : %d'%epoch)
        
        model.train()

        running_loss=0
        correct=0
        total=0

        for data in train_loader:
            
            inputs,labels=data[0].to(device),data[1].to(device)
            # forward pass
            outputs=model(inputs)
            print(outputs)
            loss=criterion(outputs,labels)

            # backward and optimise
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            
            # Compute Training Accuracy
            _, predicted_labels = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += predicted_labels.eq(labels).sum().item()
            
        train_loss=running_loss/len(train_loader)
        
        accu=100.*correct/total
        # visualisation
        writer.add_scalar("Loss/train", loss, epoch)  
        writer.add_scalar("Accuracy/train", accu, epoch)  
        
        train_accu.append(accu)
        train_losses.append(train_loss)
        print('Train Loss: %.3f | Accuracy: %.3f'%(train_loss,accu))

def test(epoch):
    model.eval()

    running_loss=0
    correct=0
    total=0

    with torch.no_grad():
        for data in valid_loader:
            features,labels=data[0].to(device),data[1].to(device)
            
            outputs=model(features)

            _, predicted_valid = torch.max(outputs.data, 1)

            loss= criterion(outputs,labels)
            running_loss+=loss.item()
            total += labels.size(0)
            correct += predicted_valid.eq(labels).sum().item()
    
    test_loss=running_loss/len(valid_loader)
    accu=100.*correct/total
    
    # visualisation
    writer.add_scalar("Loss/test", loss, epoch)  
    writer.add_scalar("Accuracy/test", accu, epoch)  

    eval_losses.append(test_loss)
    eval_accu.append(accu)

    print('Validation Loss: %.3f | Accuracy: %.3f'%(test_loss,accu))
    return test_loss

In [18]:
torch.save(model, '../models/DisfluencyNet_snd_quart_blocks.pth')

epochs=num_epochs
train_start_time = datetime.now()
for epoch in range(1,epochs+1): 
    train(epoch)
    eval_loss = test(epoch)
    torch.cuda.empty_cache()
    # if early_stopper(eval_loss, model):
    #     print('Eval performance not improving; would be early stopping now!')
    #     model.load_state_dict(early_stopper.best_model)
    #     # TODO: figure out why break makes our number of predictions go to zero :|
    #     # break

train_end_time = datetime.now()
print("\nFinished training model in " + str(train_end_time - train_start_time) + " seconds.\n")

##################################################################################################
# torch.save(model, 'DisfluencyNet_wp_quart.pth')
##################################################################################################
## TEST MODEL ##
test_loader = DataLoader(dataset=test_dataset,
                        batch_size=batch_size,
                        shuffle=True,
                        num_workers=0)

# model.load_state_dict(early_stopper.best_model)
model.eval()

with torch.no_grad():
    total = 0
    correct = 0
    n_correct = 0
    # Compute F1 score, precision and recall
    predicted_stutter = 0
    labels_stutter = 0
    correct_stutter = 0
    i = 0
    final_label = []
    final_predicted = []
    for features, labels in test_loader:
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        loss= criterion(outputs,labels)
        
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        #    final_predicted.append(predicted)
        #    final_label.append(labels)

        # predicted = torch.reshape(predicted,(outputs.shape[0],1))
        
    for i in range (0, len(predicted)) :
        # F1 score for stutter
        if (predicted[i] == 1) :
            predicted_stutter +=1
        if (labels[i] == 1) :
            labels_stutter +=1   
        if ((predicted[i] == 1) & (labels[i] == 1)):
            correct_stutter +=1
        if (predicted[i] == labels[i]) :
            n_correct = n_correct + 1

print(f'Predicted stutter: {predicted_stutter}%')
print(f'Correct stutter predictions: {correct_stutter}%')

acc_test = 100*correct/total
print(f'Accuracy of the network on test dataset is : {acc_test} %')
recall = correct_stutter/ labels_stutter
print(f'Recall of the network on test dataset is : {recall}')
precision = correct_stutter / predicted_stutter
print(f'Precision of the network on test dataset is : {precision}')
f1_score = 2 * precision * recall / (precision + recall)    
print(f'F1 Score of the network on test dataset is : {f1_score}')


Epoch : 1
{'prolongation': tensor([[ 8.1042e-02, -2.1393e-02],
        [ 7.7573e-02, -2.9791e-02],
        [ 8.0245e-02, -2.6609e-02],
        [ 5.3700e-02, -3.9686e-02],
        [ 6.2671e-02, -3.4833e-02],
        [ 6.6334e-02, -2.3358e-02],
        [ 6.2453e-02, -1.8753e-02],
        [ 6.8742e-02, -3.3702e-02],
        [ 7.0735e-02, -4.1982e-02],
        [ 8.1712e-02, -1.6882e-02],
        [ 7.9684e-02, -2.8706e-02],
        [ 7.8968e-02, -1.4025e-02],
        [ 6.6088e-02, -4.7728e-02],
        [ 5.2085e-02, -6.0981e-04],
        [ 5.4236e-02, -1.3975e-02],
        [ 4.4259e-02, -2.0116e-02],
        [ 5.8181e-02, -1.5527e-02],
        [ 8.8218e-02, -2.8902e-02],
        [ 6.1554e-02, -2.3732e-02],
        [ 5.7718e-02, -9.8951e-03],
        [ 7.5207e-02, -1.9384e-02],
        [ 6.3944e-02, -3.4848e-02],
        [ 6.5258e-02, -3.2388e-02],
        [ 4.9180e-02, -1.6077e-02],
        [ 6.3787e-02, -2.2549e-02],
        [ 7.1511e-02, -2.8834e-02],
        [ 5.3546e-02, -4.5277e-02],


TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not dict